# Phase 3 — Forecast Visual Review
Plots actual demand history against Prophet's forecast for sample products,
one from each category, so accuracy can be reviewed visually rather than
just numerically (MAPE alone can hide systematic bias, like the model
consistently under- or over-shooting the seasonal peak).

In [ ]:
import sys
import os
sys.path.append(os.path.join("..", "src", "data_generation"))
sys.path.append(os.path.join("..", "src", "forecasting"))

import pandas as pd
import matplotlib.pyplot as plt
from db import engine

%matplotlib inline

## Load accuracy summary + pick one sample product per category
Picking the median-MAPE product per category (not the best or worst) gives
a representative view of "typical" performance, not a cherry-picked one.

In [ ]:
accuracy = pd.read_sql("""
    SELECT a.product_id, p.category, a.mean_mape
    FROM forecast_accuracy_summary a
    JOIN products p ON a.product_id = p.product_id
    ORDER BY p.category, a.mean_mape
""", engine)

sample_products = {}
for category, group in accuracy.groupby("category"):
    median_row = group.iloc[len(group) // 2]
    sample_products[category] = median_row["product_id"]

print("Sample products (median MAPE per category):")
for category, product_id in sample_products.items():
    mape = accuracy[accuracy["product_id"] == product_id]["mean_mape"].iloc[0]
    print(f"  {category}: {product_id} ({mape:.2f}% MAPE)")

## Plot: full history + forecast, one chart per sample product

In [ ]:
def plot_history_and_forecast(product_id, category):
    history = pd.read_sql(
        "SELECT date as ds, units_sold as y FROM demand_history WHERE product_id = ? ORDER BY date",
        engine, params=(product_id,)
    )
    history["ds"] = pd.to_datetime(history["ds"])

    forecast = pd.read_sql(
        "SELECT ds, yhat, yhat_lower, yhat_upper FROM demand_forecasts WHERE product_id = ? ORDER BY ds",
        engine, params=(product_id,)
    )
    forecast["ds"] = pd.to_datetime(forecast["ds"])

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(history["ds"], history["y"], label="Actual demand", color="steelblue", linewidth=1)
    ax.plot(forecast["ds"], forecast["yhat"], label="Forecast", color="darkorange", linewidth=2)
    ax.fill_between(forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"],
                     color="darkorange", alpha=0.2, label="90% interval")
    ax.set_title(f"{product_id} ({category}) — Actual vs Forecast")
    ax.set_xlabel("Date")
    ax.set_ylabel("Units sold")
    ax.legend()
    plt.tight_layout()
    plt.show()

for category, product_id in sample_products.items():
    plot_history_and_forecast(product_id, category)

## Zoomed view: last 90 days of history + full forecast window
The full-history chart above compresses the forecast period into a tiny
sliver on the right. This zoomed view makes the forecast itself (and its
uncertainty band) much easier to actually evaluate.

In [ ]:
def plot_zoomed_forecast(product_id, category, lookback_days=90):
    history = pd.read_sql(
        "SELECT date as ds, units_sold as y FROM demand_history WHERE product_id = ? ORDER BY date",
        engine, params=(product_id,)
    )
    history["ds"] = pd.to_datetime(history["ds"])
    history_recent = history.tail(lookback_days)

    forecast = pd.read_sql(
        "SELECT ds, yhat, yhat_lower, yhat_upper FROM demand_forecasts WHERE product_id = ? ORDER BY ds",
        engine, params=(product_id,)
    )
    forecast["ds"] = pd.to_datetime(forecast["ds"])

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(history_recent["ds"], history_recent["y"], label="Actual (last 90 days)",
            color="steelblue", marker="o", markersize=3, linewidth=1)
    ax.plot(forecast["ds"], forecast["yhat"], label="Forecast", color="darkorange", linewidth=2)
    ax.fill_between(forecast["ds"], forecast["yhat_lower"], forecast["yhat_upper"],
                     color="darkorange", alpha=0.2, label="90% interval")
    ax.axvline(history["ds"].max(), color="gray", linestyle="--", linewidth=1, label="Forecast start")
    ax.set_title(f"{product_id} ({category}) — Zoomed: Recent History + Forecast")
    ax.set_xlabel("Date")
    ax.set_ylabel("Units sold")
    ax.legend()
    plt.tight_layout()
    plt.show()

for category, product_id in sample_products.items():
    plot_zoomed_forecast(product_id, category)

## MAPE distribution across all products, by category
A boxplot makes the category pattern (electronics harder than apparel/staples)
visually obvious in a way the printed table doesn't.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
accuracy.boxplot(column="mean_mape", by="category", ax=ax)
ax.set_title("Forecast MAPE Distribution by Category")
plt.suptitle("")
ax.set_xlabel("Category")
ax.set_ylabel("Mean MAPE (%)")
plt.tight_layout()
plt.show()

## What to look for while reviewing these charts
- **Does the forecast track the actual seasonal shape**, or does it miss the timing/magnitude of the peak?
- **Is the uncertainty band (orange shaded area) reasonable** — not so narrow it's overconfident, not so wide it's useless?
- **Any visible bias** — does the forecast consistently sit above or below actuals in the zoomed view, rather than centered on them?
- **Does the electronics product's chart visibly show a sharper, harder-to-hit peak** than apparel/staples — confirming the MAPE pattern found in Step 8?